In [1]:
import tempfile
import os
from ansys.aedt.core import Desktop, generate_unique_name
import optuna
import script_otim as script
import logging
import sys
import numpy as np

c:\Users\luis.costa\.conda\envs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
optuna.logging.get_logger("optuna").addHandler(logging.StreamHandler(sys.stdout))
study_name = "antenna_study" 
storage_name = "sqlite:///{}.db".format(study_name)

In [3]:
tmpfold = tempfile.gettempdir()

temp_folder = os.path.join(
    tmpfold,
    generate_unique_name("Antenna")
)

if not os.path.exists(temp_folder):
    os.mkdir(temp_folder)

non_graphical = os.getenv(
    "PYAEDT_NON_GRAPHICAL",
    "True"
).lower() in ("true", "1", "t")

d = Desktop(
    "2025.1",
    non_graphical=non_graphical,
    new_desktop=True
)


PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: Log on console is enabled.
PyAEDT INFO: Log on file C:\Users\LUIS~1.COS\AppData\Local\Temp\pyaedt_luis.costa_29d72214-e057-4dcc-84c6-f4899041af82.log is enabled.
PyAEDT INFO: Log on AEDT is disabled.
PyAEDT INFO: Debug logger is disabled. PyAEDT methods will not be logged.
PyAEDT INFO: Launching PyAEDT with gRPC plugin.
PyAEDT INFO: New AEDT session is starting on gRPC port 51150.
PyAEDT INFO: Electronics Desktop started on gRPC port: 51150 after 10.523864984512329 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v251\AnsysEM
PyAEDT INFO: Ansoft.ElectronicsDesktop.2025.1 version started with process ID 17368.


In [ ]:
def objective(trial):
    a1 = trial.suggest_float("a1", 1.0, 1.0)
    a2 = trial.suggest_float("a2", 1.0, 1.0)
    n1 = trial.suggest_int("n1", 1, 100)
    n2 = trial.suggest_float("n2", 1.0, 10.0)
    n3 = trial.suggest_float("n3", 1.0, 10.0)
    n4 = trial.suggest_int("n4", 1, 100)
    perc = trial.suggest_float("perc", 0.2, 0.8)
    r = trial.suggest_int("r", 90, 150)

    df = script.makeAntenna(a1, a2, n1, n2, n3, n4, perc, r, temp_folder)
    s11 = np.asarray(df['|$S_{11}$| [dB]'], dtype=float)

    proportion_bad = float(np.mean(s11 >= -10))

    return proportion_bad

study = optuna.create_study(
    directions=["minimize"],    
    study_name=study_name, 
    storage=storage_name,
    seed=123, 
    load_if_exists=True
)



[I 2025-11-24 10:48:28,198] Using an existing study with name 'antenna_study' instead of creating a new one.


Using an existing study with name 'antenna_study' instead of creating a new one.


In [ ]:
study.optimize(objective, n_trials=1)

PyAEDT INFO: Python version 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.20.1.
PyAEDT INFO: Returning found Desktop session with PID 17368!
PyAEDT INFO: Project Project742 has been created.
PyAEDT INFO: No design is present. Inserting a new design.
PyAEDT INFO: Added design 'HFSS_FXK' of type HFSS.
PyAEDT INFO: Aedt Objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m 0sec
PyAEDT INFO: 3D Modeler objects parsed. Elapsed time: 0m 0sec
PyAEDT INFO: Deleted 1 Objects: cut.
PyAEDT INFO: Parsing design objects. This operation can take time
PyAEDT INFO: Refreshing bodies from Object Info
PyAEDT INFO: Bodies Info Refreshed Elapsed time: 0m

In [ ]:
len(study.trials)

1

In [ ]:
study.best_params

{'a1': 1.0,
 'a2': 1.0,
 'n1': 23,
 'n2': 7.969403428605037,
 'n3': 1.2933486876293951,
 'n4': 68,
 'perc': 0.44166496098917285,
 'r': 114}

In [ ]:
study.best_trial

FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.5499181669394435], datetime_start=datetime.datetime(2025, 11, 5, 9, 45, 38, 449342), datetime_complete=datetime.datetime(2025, 11, 5, 10, 19, 12, 711732), params={'a1': 1.0, 'a2': 1.0, 'n1': 23, 'n2': 7.969403428605037, 'n3': 1.2933486876293951, 'n4': 68, 'perc': 0.44166496098917285, 'r': 114}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'a1': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'a2': FloatDistribution(high=1.0, log=False, low=1.0, step=None), 'n1': IntDistribution(high=100, log=False, low=1, step=1), 'n2': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n3': FloatDistribution(high=10.0, log=False, low=1.0, step=None), 'n4': IntDistribution(high=100, log=False, low=1, step=1), 'perc': FloatDistribution(high=0.8, log=False, low=0.2, step=None), 'r': IntDistribution(high=150, log=False, low=90, step=1)}, trial_id=1, value=None)

In [ ]:
d.release_desktop()

PyAEDT INFO: Desktop has been released and closed.


True